# 11 — TabPFN

**Goal:** try TabPFN (prior-fitted network) with a capped training context.

## Why this experiment?
TabPFN can be very strong on small/medium tabular problems without heavy tuning.
It has a practical context-size limit, so we use a fixed 1,024-row stratified sample.

## Approach
1. Full engineered features.
2. Take a deterministic stratified 1,024-row training context.
3. Fit TabPFN and evaluate on the full shared test set.

## What changed?
- Model: TabPFN instead of GBDT
- Training data: subsampled to 1,024 rows (by design)

## Features used in this notebook
- Full pricing + time + geo features
- Training context capped to 1,024 stratified rows (TabPFN limit)
- **How selected:** same strong FE; subsample rows, not columns
- **Why:** TabPFN is strong on medium tabular problems without heavy tuning


### Setup
Shared data load and fixed split.


In [ ]:
import os
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "shared" / "protocol.py").exists():
        EXPERIMENT_ROOT = candidate
        break
    if (candidate / "hyperack_exp" / "shared" / "protocol.py").exists():
        EXPERIMENT_ROOT = candidate / "hyperack_exp"
        break
else:
    raise RuntimeError("Run this notebook from the HyperAck project directory.")
os.chdir(EXPERIMENT_ROOT)
sys.path.insert(0, str(EXPERIMENT_ROOT))

from shared.protocol import (
    add_geo_features,
    add_pricing_features,
    add_time_features,
    base_features,
    evaluate,
    load_clean_df,
    make_xy,
    save_result,
    split_frame,
    actual_vs_predicted_report,
)

RANDOM_STATE = 42
train_df, test_df = split_frame(load_clean_df())


### Features + context
Full FE, then stratified 1,024-row context for TabPFN.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tabpfn import TabPFNClassifier

X_train, y_train = make_xy(train_df, pricing=True, time_features=True, geo=True)
X_test, y_test = make_xy(test_df, pricing=True, time_features=True, geo=True)

# Keep a deterministic, class-stratified 1,024-row context for stable local runs.
if len(X_train) > 1024:
    _, context_idx = train_test_split(
        np.arange(len(X_train)), test_size=1024, stratify=y_train, random_state=RANDOM_STATE
    )
    X_context, y_context = X_train.iloc[context_idx], y_train.iloc[context_idx]
else:
    X_context, y_context = X_train, y_train

device = "cpu"
try:
    import torch
    if torch.backends.mps.is_available():
        device = "mps"
except ImportError:
    pass
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", TabPFNClassifier(n_estimators=8, device=device, random_state=RANDOM_STATE)),
])


### Features selected and why

Columns: full engineered set (pricing + time + geo).  
Rows: a **deterministic stratified 1,024-row** training context (TabPFN context limit).

We select features by reusing the FE that already worked; we select rows by stratified sampling so class balance is preserved.

Next cell lists every selected column.


In [ ]:
feature_why = {
    "deliverey_category_id": "Delivery type — some categories get accepted more often",
    "weekday": "Day of week — weekday vs weekend courier behavior",
    "time_bucket": "Coarse time-of-day bucket from the raw data",
    "total_distance": "Trip length — longer trips can be harder to accept",
    "sum_product": "Order size / number of products",
    "source_latitude": "Pickup latitude — area effects",
    "source_longitude": "Pickup longitude — area effects",
    "destination_latitude": "Drop-off latitude — area effects",
    "destination_longitude": "Drop-off longitude — area effects",
    "first_customer_fare": "First offered customer price (usually known early)",
    "final_customer_fare": "Final customer price — strong but may be post-decision",
    "final_biker_fare": "Final courier pay — strong but may be post-decision",
    "geo_cluster": "Train-only KMeans region of the trip (pickup+drop-off)",
    "log_distance": "Log distance — softens very long trips",
    "first_fare_per_km": "First fare ÷ distance — pay vs effort",
    "final_customer_fare_per_km": "Final customer fare ÷ distance",
    "customer_fare_delta": "Final − first customer fare (price change)",
    "customer_fare_change_pct": "Relative fare change vs first offer",
    "biker_customer_gap": "Biker fare − customer fare (split / margin)",
    "biker_fare_per_km": "Courier pay per km",
    "log_final_customer_fare": "Log of final customer fare",
    "log_final_biker_fare": "Log of final biker fare",
    "hour": "Exact hour of order creation",
    "is_rush_hour": "Lunch/evening peak flag",
    "is_weekend": "Weekend flag",
    "hour_sin": "Cyclical hour (sin) so 23 is near 0",
    "hour_cos": "Cyclical hour (cos)",
    "weekday_sin": "Cyclical weekday (sin)",
    "weekday_cos": "Cyclical weekday (cos)",
    "day_of_month": "Calendar day — mild monthly pattern",
    "haversine_km": "Great-circle route distance in km",
    "latitude_delta": "North/south trip span",
    "longitude_delta": "East/west trip span",
    "geo_bearing_sin": "Trip direction (sin of bearing)",
    "geo_bearing_cos": "Trip direction (cos of bearing)",
    "distance_x_first_fare": "Interaction: long trip × price",
    "category_x_hour": "Interaction: category × hour",
    "total_distance_qbin": "Train-fitted distance quantile bin",
    "first_customer_fare_qbin": "Train-fitted first-fare quantile bin"
}

cols = list(X_train.columns)
rows = []
for c in cols:
    rows.append({
        "feature": c,
        "why_selected": feature_why.get(c, "Part of this experiment's engineered feature set"),
    })
feature_table = pd.DataFrame(rows)
print(f"Total features selected: {len(cols)}")
print("Columns:")
print(", ".join(cols))
feature_table


### Evaluate
Predict on the full held-out test set.


In [ ]:
metrics = evaluate(model, X_context, y_context, X_test, y_test)
metrics


### Actual vs predicted (test set)

After training, we score the **held-out test set** and compare:

1. **Actual** labels (`hyper_ack`) vs **predicted** labels  
2. Confusion matrix (rows = actual, columns = predicted)  
3. Per-class precision / recall / F1  
4. A sample of correct and incorrect rows with predicted probability  

This is only test-set performance — not training rows.


In [ ]:
from IPython.display import display
from shared.protocol import actual_vs_predicted_report

avp = actual_vs_predicted_report(
    metrics["y_true"],
    metrics["y_pred"],
    metrics["y_prob"],
    sample_size=25,
)
print("1) Actual vs predicted class counts")
display(avp["class_counts"])
print("2) Confusion matrix")
display(avp["confusion_matrix"])
print("3) Outcome breakdown")
display(avp["outcomes"])
print("4) Per-class metrics")
display(avp["per_class_metrics"])
print("5) Sample of actual vs predicted rows")
display(avp["prediction_sample"])


### Save result
Write experiment `11`.


In [ ]:
result_path = save_result(
    "11",
    "tabpfn",
    "TabPFN on a deterministic 1,024-row stratified context",
    metrics,
    best_model="TabPFNClassifier",
    notes="Context is capped to 1,024 rows for reliable local inference.",
    feature_count=X_train.shape[1],
)
pd.Series(metrics).drop("confusion_matrix").sort_index(), result_path


## What to look at
- ROC-AUC vs full-data trees
- Fit/predict time (TabPFN can be slower)
